# Find LSSTCamSources in all bands


---
- **Author:** Sylvie Dagoret-Campagne
- **Affiliation:** IJCLab/IN2P3/CNRS, Université Paris-Saclay
- **Created:** 2026-07-06
- **Last update:** 2026-07-06


## Imports

In [ ]:
import gc
import logging
import os
import re
import sys

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.time import Time

from lsst.daf.butler import Butler, Timespan
import lsst.geom as geom
from lsst.geom import SpherePoint, degrees

## Usefull functions

## Logging

In [ ]:
# 1. Récupérer le logger racine (ou créez un logger spécifique: logging.getLogger('mon_code'))
log = logging.getLogger()

# 2. Définir le niveau de log global (DEBUG, INFO, WARNING, ERROR)
log.setLevel(logging.INFO)

# 3. Éviter la duplication des handlers si la cellule est exécutée plusieurs fois
if not log.handlers:
    # 4. Créer un handler qui écrit vers la sortie standard (capturée par Jupyter)
    handler = logging.StreamHandler(sys.stdout)
    handler.setLevel(logging.INFO)

    # 5. Définir le format des messages (heure, niveau, nom du logger, message)
    formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
    handler.setFormatter(formatter)

    # 6. Ajouter le handler au logger
    log.addHandler(handler)

# Petit test pour vérifier que ça fonctionne
log.info("Le logging est configuré et fonctionne dans le notebook !")

## Configuration

**Edit only this cell** to point to the right Butler repository, collections,
input CSV, search radius, and output band.


In [ ]:
# ── Butler ────────────────────────────────────────────────────────────────
repo = "dp2_prep"

collection = [
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage1",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage2",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage3",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage4",
]

instrument = "LSSTCam"
skymapName = "lsst_cells_v2"

BANDSEL = "r"

# ── Input targets ─────────────────────────────────────────────────────
target_file = "summary_visit_counts_per_star_V17-21_r2.0deg.csv"

# ── Cross-match search radius ─────────────────────────────────────────────
MATCH_RADIUS_ARCSEC = 1.0  # maximum separation for a valid match [arcsec]

DATE_START = "2025-04-01T00:00:00"
DATE_STOP = "2026-07-01T00:00:00"
time_start = Time(DATE_START, format="isot", scale="utc")
time_stop = Time(DATE_STOP, format="isot", scale="utc")
MJD_START = time_start.mjd
MJD_STOP = time_stop.mjd
DELTAMJD_DAYS = MJD_STOP - MJD_START
log.debug(f"MJD ::: start = {MJD_START} , stop = {MJD_STOP} , delta t = {DELTAMJD_DAYS} days")

SRC_COLUMNS = [
    "coord_ra",
    "coord_dec",
    "parentSourceId",
    "x",
    "y",
    "xErr",
    "yErr",
    "ra",
    "dec",
    "raErr",
    "decErr",
    "calibFlux",
    "calibFluxErr",
    # "ap03Flux",
    # "ap03FluxErr",
    # "ap03Flux_flag",
    # "ap06Flux",
    # "ap06FluxErr",
    # "ap06Flux_flag",
    "ap09Flux",
    "ap09FluxErr",
    "ap09Flux_flag",
    "ap12Flux",
    "ap12FluxErr",
    "ap12Flux_flag",
    "ap17Flux",
    "ap17FluxErr",
    "ap17Flux_flag",
    "ap25Flux",
    "ap25FluxErr",
    "ap25Flux_flag",
    "ap35Flux",
    "ap35FluxErr",
    "ap35Flux_flag",
    # "ap50Flux",
    # "ap50FluxErr",
    # "ap50Flux_flag",
    # "ap70Flux",
    # "ap70FluxErr",
    # "ap70Flux_flag",
    "sky",
    "skyErr",
    "psfFlux",
    "psfFluxErr",
    #'ixx','iyy','ixy','ixxPSF','iyyPSF','ixyPSF','ixxDebiasedPSF','iyyDebiasedPSF','ixyDebiasedPSF',
    #'gaussianFlux','gaussianFluxErr',
    "extendedness",
    "sizeExtendedness",
    #'blendedness_abs','blendedness_flag','blendedness_flag_noCentroid','blendedness_flag_noShape',
    "apFlux_12_0_flag",
    # "apFlux_12_0_flag_apertureTruncated",
    # "apFlux_12_0_flag_sincCoeffsTruncated",
    "apFlux_12_0_instFlux",
    "apFlux_12_0_instFluxErr",
    "apFlux_17_0_flag",
    "apFlux_17_0_instFlux",
    "apFlux_17_0_instFluxErr",
    "apFlux_35_0_flag",
    "apFlux_35_0_instFlux",
    "apFlux_35_0_instFluxErr",
    # "apFlux_50_0_flag",
    # "apFlux_50_0_instFlux",
    # "apFlux_50_0_instFluxErr",
    #'normCompTophatFlux_flag','normCompTophatFlux_instFlux','normCompTophatFlux_instFluxErr',
    "extendedness_flag",
    "sizeExtendedness_flag",
    #'footprintArea_value','invalidPsfFlag','jacobian_flag','jacobian_value',
    "localBackground_instFlux",
    "localBackground_instFluxErr",
    "localBackground_flag",
    # "localBackground_flag_noGoodPixels",
    # "localBackground_flag_noPsf",
    #'pixelFlags_bad','pixelFlags_cr','pixelFlags_crCenter','pixelFlags_edge','pixelFlags_interpolated','pixelFlags_interpolatedCenter','pixelFlags_nodata','pixelFlags_offimage','pixelFlags_saturated','pixelFlags_saturatedCenter','pixelFlags_suspect','pixelFlags_suspectCenter',
    #'psfFlux_apCorr','psfFlux_apCorrErr',
    #'psfFlux_area','psfFlux_flag','psfFlux_flag_apCorr','psfFlux_flag_edge','psfFlux_flag_noGoodPixels',
    #'gaussianFlux_flag','centroid_flag','centroid_flag_almostNoSecondDerivative','centroid_flag_badError','centroid_flag_edge','centroid_flag_noSecondDerivative','centroid_flag_notAtMaximum','centroid_flag_resetToPeak',
    #'variance_flag','variance_flag_emptyFootprint','variance_value',
    #'calib_astrometry_used','calib_photometry_reserved','calib_photometry_used','calib_psf_candidate','calib_psf_reserved','calib_psf_used',
    #'deblend_deblendedAsPsf','deblend_hasStrayFlux','deblend_masked','deblend_nChild','deblend_parentTooBig','deblend_patchedTemplate','deblend_rampedTemplate','deblend_skipped','deblend_tooManyPeaks',
    #'hsmPsfMoments_flag','hsmPsfMoments_flag_no_pixels','hsmPsfMoments_flag_not_contained','hsmPsfMoments_flag_parent_source',
    #'iDebiasedPSF_flag','iDebiasedPSF_flag_no_pixels','iDebiasedPSF_flag_not_contained','iDebiasedPSF_flag_parent_source','iDebiasedPSF_flag_galsim','iDebiasedPSF_flag_edge',
    #'hsmShapeRegauss_flag','hsmShapeRegauss_flag_galsim','hsmShapeRegauss_flag_no_pixels','hsmShapeRegauss_flag_not_contained','hsmShapeRegauss_flag_parent_source',
    "sky_source",
    "visit",
    "detector",
    "band",
    "physical_filter",
    "sourceId",
]

log.info("Butler configuration done.")

## DDF selected

In [ ]:
# LSST Deep Drilling Fields (RA/Dec J2000)
DEEP_FIELDS = {
    "COSMOS": (150.1191, 2.2058),
    "ELAIS-S1": (9.4500, -44.000),
    "XMM-LSS": (35.7080, -4.750),
    "ECDFS": (53.1250, -27.8),
    "EDFS-a": (58.9, -49.315),
    "EDFS-b": (63.6, -47.6),
    "EDFS": (61.24, -48.423),
    "M49": (187.4, 8.0),
}

## Initialise the Butler

In [ ]:
butler = Butler(repo, collections=collection)
registry = butler.registry
skymap = butler.get("skyMap", skymap=skymapName, collections=collection)
log.info(f"Butler initialised | repo: {repo}")

## Which visits

In [ ]:
visits = butler.registry.queryDimensionRecords("visit")

target_names = sorted({v.target_name for v in visits if v.target_name is not None})
print(target_names)

In [ ]:
ddf_keywords = ["cosmos", "ecdfs", "cdfs", "xmm", "elaiss1", "elais", "edfs"]


def is_ddf(name):
    n = name.lower()
    return any(k in n for k in ddf_keywords)


ddf_names = sorted({name for name in target_names if is_ddf(name)})

for n in ddf_names:
    print(n)

In [ ]:
def normalize_ddf(name):
    n = name.lower()
    if "cosmos" in n:
        return "COSMOS"
    if "ecdfs" in n or "cdfs" in n:
        return "ECDFS"
    if "xmm" in n:
        return "XMM-LSS"
    if "elaiss1" in n or "elais" in n:
        return "ELAIS-S1"
    if "edfs" in n:
        return "EDFS"
    return None


unique_ddf = sorted({normalize_ddf(name) for name in target_names if normalize_ddf(name) is not None})

print(unique_ddf)

In [ ]:
from collections import Counter

counter = Counter(normalize_ddf(v.target_name) for v in visits if normalize_ddf(v.target_name))

for k, v in counter.items():
    print(k, v)

### Retrieve all Visits in all DDF

In [ ]:
visits = list(butler.registry.queryDimensionRecords("visit"))

ddf_visits = {"COSMOS": [], "XMM-LSS": [], "EDFS": [], "ECDFS": [], "ELAIS-S1": []}

for v in visits:
    name = normalize_ddf(v.target_name)
    if name:
        ddf_visits[name].append(v.id)

## Auto-discover the object-table dataset type

The catalogue dataset name changed across pipeline versions:

| Pipeline era | Dataset type name          |
|:-------------|:---------------------------|
| Gen2 / HSC   | `deepCoadd_obj`            |
| DP1          | `objectTable_tract`        |
| DP2+         | `object_table_tract`       |

We probe the registry so the notebook is collection-agnostic.


In [ ]:
# Prioritised list of candidate object-table dataset type names
SRC_TABLE_CANDIDATES = [
    "source",
    "sourceTable," "sourceTable_visit",  # visit-level source table (fallback)
]

# List all source-table-related types actually in the registry
all_src_types = [
    d.name for d in registry.queryDatasetTypes() if "source" in d.name.lower() or "table" in d.name.lower()
]
# print("src-table-related dataset types in registry:")
# for t in sorted(all_src_types):
#    print(f"  {t}")

# Pick the first candidate that is registered
SRC_DATASET = None
for name in SRC_TABLE_CANDIDATES:
    if dataset_type_exists(butler, name):
        SRC_DATASET = name
        log.info(f"\n✔ Selected src-table dataset type: '{SRC_DATASET}'")
        break

if SRC_DATASET is None:
    raise RuntimeError(
        "No recognised source-table dataset type found in this Butler collection. "
        f"Candidate types seen: {all_src_types}"
    )

In [ ]:
refsall = list(butler.query_datasets("source", where="band = 'y'"))

# refs = [r for r in refsall if r.dataId["visit"] in ddf_visits["COSMOS"]]

In [ ]:
list_of_refs_per_ddf = {}

for ddf_name in unique_ddf:
    refs = [r for r in refsall if r.dataId["visit"] in ddf_visits[ddf_name]]
    list_of_refs_per_ddf[ddf_name] = refs


for ddf_name in unique_ddf:
    print(ddf_name, len(list_of_refs_per_ddf[ddf_name]))

In [ ]:
# Load the first source table to probe the schema
probe_refs = list_of_refs_per_ddf["COSMOS"]
df_probe = butler.get(probe_refs[0], parameters={"columns": SRC_COLUMNS})
if not isinstance(df_probe, pd.DataFrame):
    df_probe = df_probe.to_pandas()
log.info(f"Probe source table: {len(df_probe)} rows, {len(df_probe.columns)} columns")

In [ ]:
logging.info(f"Columns of sources dataframe : {df_probe.columns.tolist()}")